In [16]:
import os 
import geopandas as gpd

print("GeoPandas version:", gpd.__version__)

GeoPandas version: 1.0.1


Load the hydrant and neighborhood datasets, then inspect their initial coordinate reference systems.

In [38]:
# Read the source GeoJSON files
hydrants = gpd.read_file('./data/raw/hydrants.geojson')
hydrants = hydrants[['boro', 'latitude', 'longitude', 'geometry']]

neighborhoods = gpd.read_file('./data/raw/neighborhoods.geojson')
neighborhoods = neighborhoods[['boroname', 'borocode','ntaname','nta2020', 'countyfips', 'geometry']]

print('*' * 60)
print('Hydrants shape', hydrants.shape, 'CRS', hydrants.crs)
print('Neighborhoods shape', neighborhoods.shape, 'CRS', neighborhoods.crs)

************************************************************
Hydrants shape (109725, 4) CRS EPSG:4326
Neighborhoods shape (262, 6) CRS EPSG:4326


Convert both GeoDataFrames to the same projected CRS so spatial operations are accurate.

In [47]:
# Inspect unique neighborhood names before projection
print('Sample neighborhood values:', neighborhoods['ntaname'].unique()[:10])

hydrants = hydrants.to_crs(epsg=2263)
neighborhoods = neighborhoods.to_crs(epsg=2263)

print('Hydrants shape', hydrants.shape, 'CRS', hydrants.crs)
print('Neighborhoods shape', neighborhoods.shape, 'CRS', neighborhoods.crs)

Sample neighborhood values: ['Greenpoint' 'Williamsburg' 'South Williamsburg' 'East Williamsburg'
 'Brooklyn Heights' 'Downtown Brooklyn-DUMBO-Boerum Hill' 'Fort Greene'
 'Clinton Hill' 'Brooklyn Navy Yard' 'Bedford-Stuyvesant (West)']
Hydrants shape (109725, 4) CRS EPSG:2263
Neighborhoods shape (262, 6) CRS EPSG:2263


Perform a spatial join to attach neighborhood attributes to each hydrant, then count hydrants per neighborhood.

In [66]:
# Spatially join hydrants to the neighborhood polygons
hydrants_with_neighborhood = gpd.sjoin(hydrants, neighborhoods, how='left', predicate='within')

# print(hydrants_with_neighborhood.head())
print(f"Hydrants with neighborhood info: {hydrants_with_neighborhood.shape}")

# Group by neighborhood and count hydrants per neighborhood
hydrant_counts = hydrants_with_neighborhood.groupby('ntaname').size().reset_index(name='hydrant_count')

#hydrants which didn't fall inside any polygon
unmatched = hydrants_with_neighborhood[hydrants_with_neighborhood['index_right'].isna()]
print("hydrants which didn't fall inside any polygon :",len(unmatched))

unmatched[['latitude', 'longitude']].head(10)

hydrant_counts

Hydrants with neighborhood info: (109725, 11)
hydrants which didn't fall inside any polygon : 31


,ntaname,hydrant_count
0,Allerton,280
1,Alley Pond Park,48
2,Annadale-Huguenot-Prince's Bay-Woodrow,1708
3,Arden Heights-Rossville,717
4,Astoria (Central),348
...,...,...
254,Windsor Terrace-South Slope,305
255,Woodhaven,516
256,Woodlawn Cemetery,13
257,Woodside,566


Hydrant Density 

In [72]:
SQ_FT_PER_KM2 = 10_763_910.42

neighborhoods['Area_km2']=(neighborhoods.geometry.area)/SQ_FT_PER_KM2

result=neighborhoods.merge(hydrant_counts,on='ntaname',how='left').fillna({"hydrant_count": 0})

result['Density_perSqkm']=(result['hydrant_count']/result['Area_km2'])
result.sort_values("Density_perSqkm", ascending=False)

,boroname,borocode,ntaname,nta2020,countyfips,geometry,Area_km2,hydrant_count,Density_perSqkm
132,Manhattan,1,Gramercy,MN0602,061,"MULTIPOLYGON (((990196.892 207745.371, 990187....",0.699188,269.0,384.731873
121,Manhattan,1,SoHo-Little Italy-Hudson Square,MN0201,061,"MULTIPOLYGON (((983469.159 204638.902, 983496....",1.200006,432.0,359.998077
119,Manhattan,1,Tribeca-Civic Center,MN0102,061,"MULTIPOLYGON (((984440.604 200699.422, 984402....",1.261461,433.0,343.252757
123,Manhattan,1,West Village,MN0203,061,"MULTIPOLYGON (((981713.541 209788.14, 981751 2...",1.339370,447.0,333.738876
118,Manhattan,1,Financial District-Battery Park City,MN0101,061,"MULTIPOLYGON (((984032.884 192223.749, 983984....",1.786223,570.0,319.109090
...,...,...,...,...,...,...,...,...,...
68,Brooklyn,3,Shirley Chisholm State Park,BK5693,047,"MULTIPOLYGON (((1019419.268 173893.122, 101946...",1.698704,2.0,1.177368
258,Staten Island,5,Fort Wadsworth,SI9561,085,"MULTIPOLYGON (((967656.829 155637.132, 967549....",0.916692,1.0,1.090879
13,Brooklyn,3,The Evergreens Cemetery,BK0471,047,"MULTIPOLYGON (((1012948.8 187911.295, 1012925....",0.518903,0.0,0.000000
67,Brooklyn,3,Jamaica Bay (West),BK5692,047,"MULTIPOLYGON (((1022227.32 152028.146, 1022078...",3.693179,0.0,0.000000
